# 08 – Full Comparison: MARL vs Single-Agent vs Traditional ML (RQ1)

This notebook evaluates all approaches on the same test set and produces the main comparison table for Research Question 1.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("..")
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
X = np.load(DATA_PROCESSED / "X_fused.npy")
y = np.load(DATA_PROCESSED / "y.npy")
thin = np.load(DATA_PROCESSED / "thin.npy")

X_train, X_test, y_train, y_test, thin_train, thin_test = train_test_split(
    X, y, thin, test_size=0.25, random_state=42, stratify=y
)

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, state_dim, n_actions=2):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(state_dim, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU())
        self.actor = nn.Linear(64, n_actions)
        self.critic = nn.Linear(64, 1)
    def forward(self, x):
        feat = self.shared(x)
        return self.actor(feat), self.critic(feat)

class MultiAgentCoordinator(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.risk_agent = nn.Sequential(nn.Linear(state_dim, 64), nn.ReLU(), nn.Linear(64, 2))
        self.fairness_agent = nn.Sequential(nn.Linear(state_dim, 64), nn.ReLU(), nn.Linear(64, 2))
        self.portfolio_agent = nn.Sequential(nn.Linear(state_dim, 64), nn.ReLU(), nn.Linear(64, 2))
        self.combine = nn.Sequential(nn.Linear(6, 32), nn.ReLU(), nn.Linear(32, 2))
    def forward(self, x):
        r = self.risk_agent(x)
        f = self.fairness_agent(x)
        p = self.portfolio_agent(x)
        return self.combine(torch.cat([r, f, p], dim=-1))

ppo = ActorCritic(X.shape[1]).to(device)
ppo_path = RESULTS / "single_agent_ppo.pt"
if ppo_path.exists():
    ppo.load_state_dict(torch.load(ppo_path, map_location=device))
ppo.eval()

marl = MultiAgentCoordinator(X.shape[1]).to(device)
marl_path = RESULTS / "multi_agent_coordinator.pt"
if marl_path.exists():
    marl.load_state_dict(torch.load(marl_path, map_location=device))
marl.eval()

In [ ]:
def evaluate_policy(model, X_te, y_te, thin_te, is_marl=False):
    preds = []
    with torch.no_grad():
        for i in range(len(X_te)):
            state = torch.tensor(X_te[i], dtype=torch.float32, device=device)
            if is_marl:
                logits = model(state)
            else:
                logits, _ = model(state)
            action = logits.argmax().item()
            preds.append(action)
    preds = np.array(preds)
    
    auc = roc_auc_score(y_te, preds) if len(np.unique(preds)) > 1 else 0.5
    f1 = f1_score(y_te, preds, zero_division=0)
    rec = recall_score(y_te, preds, zero_division=0)
    prec = precision_score(y_te, preds, zero_division=0)
    
    mask = thin_te == 1
    thin_approval = (preds[mask] == 0).mean() if mask.sum() > 0 else 0
    
    return {
        "AUC": auc, "F1": f1, "Recall": rec, "Precision": prec,
        "Thin-file Approval Rate": thin_approval
    }

In [ ]:
results = []

lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(X_train, y_train)
proba_lr = lr.predict_proba(X_test)[:, 1]
pred_lr = (proba_lr >= 0.5).astype(int)
results.append({
    "Model": "Logistic Regression",
    "AUC": roc_auc_score(y_test, proba_lr),
    "F1": f1_score(y_test, pred_lr),
    "Recall": recall_score(y_test, pred_lr),
    "Precision": precision_score(y_test, pred_lr),
    "Thin-file Approval Rate": (pred_lr[thin_test==1] == 0).mean()
})

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]
pred_rf = (proba_rf >= 0.5).astype(int)
results.append({
    "Model": "Random Forest",
    "AUC": roc_auc_score(y_test, proba_rf),
    "F1": f1_score(y_test, pred_rf),
    "Recall": recall_score(y_test, pred_rf),
    "Precision": precision_score(y_test, pred_rf),
    "Thin-file Approval Rate": (pred_rf[thin_test==1] == 0).mean()
})

results.append({"Model": "Single-Agent PPO", **evaluate_policy(ppo, X_test, y_test, thin_test)})
results.append({"Model": "Multi-Agent MARL", **evaluate_policy(marl, X_test, y_test, thin_test, is_marl=True)})

res_df = pd.DataFrame(results)
print(res_df.round(3).to_string(index=False))
res_df.to_csv(RESULTS / "full_comparison.csv", index=False)
print("\nSaved → results/full_comparison.csv")